# DGC / Waddington-OT TF-control analysis (self-contained)

Depends only on the `dgc_pipeline/` package and standard scientific Python
(numpy, scipy, pandas, anndata).

Pipeline:
1.  Backend + data load
2.  Trajectories (option A: load cache / option B: build from transport maps + save)
3.  Mouse panel symbols  <-- the gene-name list both pieces of the analysis live in
4.  Operators: 12h M and the interleaved 24h M (with the M_24 vs M_12^2 check)
5.  Target (iPSC biological)
6.  States in CENTERED coordinates (M was fit on centered dynamics)
7.  Binding-B  (the object DGC uses)
8.  b_OSKM_effect ground truth (inferred from the known Dox input in Schiebinger)
9.  Norman-B (CRISPRa Perturb-seq deconvolution against the 24h operator)
10. **Head-to-head**: binding-B-OSKM vs Norman-B-OSKM, both against b_OSKM_effect
11. Scoring + best-day control
12. Importance (LOO + mass)
13. Diagnostics (fusion vs averaging, forcing SVD, geometry)

**Frame convention:** M is fit on mean-centered dynamics; the gap
`(x_target - xbar) - M^N (x_init - xbar)` is the centered residual. Score in
centered coordinates throughout.

## 0. Setup

In [2]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath('.'))   # folder containing dgc_pipeline/

from dgc_pipeline import (Backend, OperatorSequence, FusedOperatorBuilder,
                          DGCOperatorBuilder, TargetBuilder, TrajectoryBuilder,
                          BOnlyScorer, DynamicsScorer, ControlBestDayScorer,
                          InterleavedFusion, NormanBBuilder,
                          make_response_sum, tikhonov_deconvolve,
                          fusion_vs_averaging, forcing_svd, forcing_geometry)
from dgc_pipeline import io as dio

DATA = dict(
    expr      = 'data/ExprMatrix.var.genes.h5ad',
    cell_days = 'data/cell_days.txt',
    cell_sets = 'major_cell_sets.gmt',
    tmap_dir  = 'tmaps',
    b_cache   = 'b_cache/B_binding_07a8161f4b109116.npz',
    traj_cache= 'traj_cache.npz',
    norman    = 'data/fibroblast_CRISPRa_custom_mean.h5ad',  # from zenodo.org/records/15200179
    norman_out= 'b_norman.npz',
)
USE_GPU = True
IPSC_SET = 'iPSC'

## 1. Backend

In [3]:
be = (Backend.gpu(require=True) if USE_GPU else Backend.cpu()).activate()
print('on_gpu:', be.on_gpu)

  GPU backend: NVIDIA GeForce RTX 5060 Laptop GPU (compute 12.0), kernel test OK
on_gpu: True


## 2. Load expression, cell days, transport maps

In [5]:
A = dio.load_expression(DATA['expr'])
genes = np.asarray(A.var_names)
cell_days = dio.load_cell_days(DATA['cell_days'])
tmap_chain = dio.discover_tmaps(DATA['tmap_dir'])
print(f'genes={len(genes)}  tmaps={len(tmap_chain)}  cells-with-days={len(cell_days)}')

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/ExprMatrix.var.genes.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## 3. Trajectories: 3A (load) OR 3B (build + save)

### 3A. Load cached

In [ ]:
if os.path.exists(DATA['traj_cache']):
    Z = TrajectoryBuilder.load(DATA['traj_cache'])
    traj_c, xbar, panel = Z['traj_c'], Z['xbar'], Z['panel']
    seed_ids = list(Z['seed_ids']) if 'seed_ids' in Z else None
    fate     = Z['fate'] if 'fate' in Z else None
    L, K, G = traj_c.shape; n_steps = K - 1
    print(f'loaded: L={L} K={K} G={G}; panel={len(panel)}')
else:
    print('no cache yet -- run 3B')

loaded: L=4556 K=39 G=1479; panel=1479


### 3B. Build from transport maps + save

In [ ]:
tbuild = TrajectoryBuilder(tmap_chain, A, genes)
traj, seed_ids = tbuild.build()
traj_c, xbar   = TrajectoryBuilder.center(traj)
panel          = TrajectoryBuilder.panel(traj, n_top=None)   # all genes by default
tb_t = TargetBuilder(A, genes, DATA['cell_sets'], ipsc_key_substr=IPSC_SET)
ipsc_rows = tb_t._resolve_rows()
ipsc_cell_ids = np.asarray(A.obs_names)[ipsc_rows] if len(ipsc_rows) else []
fate = tbuild.seed_fate(seed_ids, ipsc_cell_ids) if len(ipsc_cell_ids) else None
L, K, G = traj_c.shape; n_steps = K - 1
TrajectoryBuilder.save(DATA['traj_cache'], traj_c, xbar, panel,
                       seed_ids=seed_ids, fate=fate)
print(f'built + saved: L={L} K={K} G={G}; panel={len(panel)}')

  expected-path trajectories: L=4556 seeds, K=39 days, G=1479
  trajectory cache saved -> traj_cache.npz
built + saved: L=4556 K=39 G=1479; panel=1479


## 4. Mouse panel symbols
The list of mouse gene symbols for the panel -- this is the gene-name list every
downstream piece (orthology, binding-B subset, Norman-B subset) lives in. It is
just `genes[panel]`. Defined here once so it can be referenced everywhere.

In [ ]:
mouse_panel_symbols = np.asarray([str(g) for g in genes[panel]])
print(f'mouse_panel_symbols: {len(mouse_panel_symbols)} genes')
print(f'  first 10: {list(mouse_panel_symbols[:10])}')
# spot-check OSKM is in the panel
for tf in ['Pou5f1','Sox2','Klf4','Myc','Nanog','Esrrb','Zfp42','Nr5a2']:
    print(f'  {tf}: {"YES" if tf in mouse_panel_symbols else "no"}')

mouse_panel_symbols: 1479 genes
  first 10: [np.str_('Fam150a'), np.str_('3110035E14Rik'), np.str_('Prex2'), np.str_('Sulf1'), np.str_('Pi15'), np.str_('Crispld1'), np.str_('Tfap2b'), np.str_('Khdc1c'), np.str_('Col9a1'), np.str_('Il1r2')]
  Pou5f1: no
  Sox2: YES
  Klf4: no
  Myc: YES
  Nanog: YES
  Esrrb: no
  Zfp42: no
  Nr5a2: no


## 5. Operators: 12h M_12, 24h M_24 (interleaved), and the consistency check
We fit BOTH a 12h-step operator (for fine-grained scoring) AND a 24h-step
operator (for the deconvolution -- S_7 is much better-conditioned than S_14).

The PRIMARY consistency check the InterleavedFusion prints is `cos(D_24_fused,
M_12^2 - I)`: the 24h fit and the squared-12h-fit should agree if the dynamics
are time-translation invariant. M1 vs M2 disagreement alone is NOT evidence
against the model -- period-2 effects can split them benignly.

In [ ]:
# 12h operator (as before)
fused_op_12, info12 = FusedOperatorBuilder(be).build(
    traj_c, ridge=0.0, eps_fuse=0.05, gene_panel=panel, n_steps_hint=n_steps)
print(f"\nrho(M_12)={info12['rho']:.4f}  rank(D_12)={info12['fused_rank']}")

# DGC rank-one (for like-for-like comparison)
dgc_ops, _ = DGCOperatorBuilder().build(traj_c[:, :, panel])
seq_dgc    = OperatorSequence(dgc_ops)
seq_fused  = OperatorSequence([fused_op_12])

  [Plot] Individual percentage scree plot saved to scree_plot.png
  fused over 4556 lineages (ridge=0, max_rank=auto(~snap/2)); per-lineage rank ~19; rank(D_fused)=1418 (G=1479)
  FUSED global operator fit R^2: 0.869
  FUSED per-lineage fit R^2: median=0.871 mean=0.869 [10th=0.861, 90th=0.878]
  SIGNAL VARIANCE: State Difference Var(DX) = 6.1911e-02 | Residual Var(b_k) = 8.0960e-03
                   Residual is 13.08% of total state variance
  CROSS-SECTIONAL FORCING RANK: Top 10 Mean SVs: [207.4, 21.8, 7.5, 5.7, 4.5, 3.5, 3.0, 2.8, 2.6, 2.4]
                                Top 4 components capture 99.63% of forcing variance
  per-COMPONENT propagator rho(I+A_l): median=0.957 mean=0.955 max=1.028 min=0.898; 146/4556 unstable (rho>1)
  S=sum P_l spectrum: min=0.00 max=4554.79 mean=62.55 | eps_fuse=0.05
  PROPAGATOR M=I+D spectrum: rho(M)=1.0041  (UNSTABLE -> powers explode)
    sigma_max(M)=23.8123  (non-normality gap sigma_max-rho = 22.8082; large => transient amplification)
    |eig(

In [ ]:
# 24h interleaved operator, with the M_fused vs M_12^2 primary check
ifu = InterleavedFusion(FusedOperatorBuilder(be))
op_24, info24 = ifu.build(traj_c, gene_panel=panel, M_12_op=fused_op_12,
                          ridge=0.0, eps_fuse=0.05,
                          n_steps_hint=n_steps // 2)
# WATCH the 'PRIMARY CHECK' line above: cos(D_24, M_12^2 - I)
# >0.9 = great; 0.7-0.9 = usable with some bias; <0.7 = something's off.

  fitting M1 on even timepoints  (K_even=20)
  [Plot] Individual percentage scree plot saved to scree_plot.png
  fused over 4556 lineages (ridge=0, max_rank=auto(~snap/2)); per-lineage rank ~9; rank(D_fused)=1355 (G=1479)
  FUSED global operator fit R^2: 0.886
  FUSED per-lineage fit R^2: median=0.886 mean=0.886 [10th=0.880, 90th=0.891]
  SIGNAL VARIANCE: State Difference Var(DX) = 1.0688e-01 | Residual Var(b_k) = 1.2203e-02
                   Residual is 11.42% of total state variance
  CROSS-SECTIONAL FORCING RANK: Top 10 Mean SVs: [263.2, 24.4, 6.8, 5.0, 3.9, 3.0, 2.6, 2.3, 2.1, 1.9]
                                Top 4 components capture 99.87% of forcing variance
  per-COMPONENT propagator rho(I+A_l): median=0.887 mean=0.891 max=0.949 min=0.850; 0/4556 unstable (rho>1)
  S=sum P_l spectrum: min=0.00 max=4551.84 mean=30.83 | eps_fuse=0.05
  PROPAGATOR M=I+D spectrum: rho(M)=1.0005  (UNSTABLE -> powers explode)
    sigma_max(M)=9.7570  (non-normality gap sigma_max-rho = 8.7565; lar

## 6. Target -- iPSC biological state (purity-gated)

In [ ]:
days_arr = sorted(set(cell_days.values()))
tb = TargetBuilder(A, genes, DATA['cell_sets'], ipsc_key_substr=IPSC_SET,
                   cell_days=cell_days, final_day=max(days_arr))
ipsc_full = tb.biological()
if ipsc_full is None:
    print('  biological() returned None; falling back to centroid()')
    ipsc_full = tb.centroid()
if ipsc_full is None:
    print('  centroid() also None; using mean trajectory endpoint as fallback')
    ipsc_full = traj_c.mean(axis=0)[-1] + xbar
print(f'  target source: ||ipsc_full||={np.linalg.norm(ipsc_full):.1f}')

  biological() returned None; falling back to centroid()
  centroid() also None; using mean trajectory endpoint as fallback
  target source: ||ipsc_full||=31.0


## 7. States (CENTERED -- matches the operator frame)

In [ ]:
x_init   = traj_c.mean(axis=0)[0, panel]
x_target = (ipsc_full - xbar)[panel]
print(f'||x_init||={np.linalg.norm(x_init):.1f}  ||x_target||={np.linalg.norm(x_target):.1f}')

||x_init||=24.1  ||x_target||=11.8


## 8. Binding-B (the existing DGC-style construction, under test)

In [ ]:
Bz = np.load(DATA['b_cache'], allow_pickle=True)
B_full   = Bz['B'] if 'B' in Bz else Bz[Bz.files[0]]
tf_names_binding = (list(Bz['tf_names']) if 'tf_names' in Bz
                    else [f'TF{j}' for j in range(B_full.shape[1])])
B_binding = B_full[panel] if B_full.shape[0] != len(panel) else B_full
tf_filter_binding = [j for j in range(B_binding.shape[1])
                     if np.linalg.norm(B_binding[:, j]) > 1e-12]
name_binding = lambda j: tf_names_binding[j]
print(f'binding-B panel {B_binding.shape}; scorable: {len(tf_filter_binding)}')

binding-B panel (1479, 884); scorable: 57


## 9. Ground truth: `b_OSKM_effect` from the known Dox input
The empirically recovered OSKM forcing direction in your data. NOT a circular
test of itself -- it's the *ground truth* both binding-B and Norman-B will be
judged against in cell 10.

In [ ]:
M12 = fused_op_12.as_matrix(len(panel))
meanc = traj_c[:, :, panel].mean(axis=0)
days  = np.array([0.5*k for k in range(n_steps)])
u_dox = np.ones(n_steps)   # constant Dox over the horizon (your data showed Bu ~ flat)
# u_dox = (days < 16.0).astype(float)   # alternative: Dox-off at day 16
resid_steps    = np.array([meanc[k+1] - M12 @ meanc[k] for k in range(n_steps)])
b_OSKM_effect  = (resid_steps * u_dox[:, None]).sum(axis=0) / ((u_dox*u_dox).sum() + 1e-12)
print(f'||b_OSKM_effect|| = {np.linalg.norm(b_OSKM_effect):.4f}')

||b_OSKM_effect|| = 0.0821


## 10. Norman-B from CRISPRa Perturb-seq + 24h deconvolution
Download `fibroblast_CRISPRa_mean_pop.h5ad` (1.7 GB) from
https://zenodo.org/records/15200179 and place it at `DATA['norman']`.

The build is end-to-end: per-TF Delta_x in Hs27 (human fibroblast) cells -->
orthology map to mouse panel --> Tikhonov deconvolution by S_7 (where S_7 uses
the 24h operator from cell 5).

In [ ]:
import os

if os.path.exists(DATA['norman']):
    # Point to the custom mean file you just generated
    norman = NormanBBuilder('data/fibroblast_CRISPRa_custom_mean.h5ad')
    
    # --- THE FIX: Update metadata pointers for the new file ---
    norman.detect_perturbation_column = lambda: 'guide_target'
    norman.control_keywords = ['non'] # The single-cell file uses 'non-targeting' or 'non'
    # ----------------------------------------------------------
    
    humanized_panel_symbols = [g.upper() for g in mouse_panel_symbols]
    
    B_norman, tf_names_norman, Delta_raw, info_norman = norman.build(
        M_24h_D=op_24.D,
        mouse_panel_names=humanized_panel_symbols,  
        n_steps_to_harvest=7,           
        tikhonov_frac=1e-2,
        min_cells_per_tf=1,
    )
    
    NormanBBuilder.save(DATA['norman_out'], B_norman, tf_names_norman,
                        mouse_panel_symbols, Delta=Delta_raw, info=info_norman)
                        
    print(f"\nB_norman shape {B_norman.shape}; {info_norman['n_TF']} TFs; "
          f"panel coverage {info_norman['n_panel_covered']}/{info_norman['G_panel']}")

  loading data/fibroblast_CRISPRa_custom_mean.h5ad ...
    shape: 1837 rows x 30395 genes
    .obs columns: ['guide_target']
    controls: 2 rows (0.1% of total)
    per-TF delta: 1835 TFs with >= 1 cells, in 30395 human genes
    orthology: 0/30395 human rows mapped to 0/1479 mouse panel rows (0.0%)
    [warn] 30395 human rows had no panel ortholog (dropped). pass orthology_map to extend coverage.

  S_7 spectrum: sigma_max=45.664  sigma_min=2.140e-01  cond=2.134e+02  effective_rank=1474/1479
  Tikhonov lambda = 2.085e+01 (=0.01 x sigma_max^2)
  saved B -> b_norman.npz  (shape (1479, 1835), 1835 TFs)

B_norman shape (1479, 1835); 1835 TFs; panel coverage 0/1479


In [ ]:
import anndata as ad

adata_test = ad.read_h5ad(DATA['norman'])

print("Available gene metadata columns (adata.var):")
print(adata_test.var.columns.tolist())

print("\nFirst 10 items in the index (adata.var_names):")
print(adata_test.var_names[:10].tolist())

Available gene metadata columns (adata.var):
[]

First 10 items in the index (adata.var_names):
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [ ]:
import anndata as ad

print(f"Loading raw Norman dataset from {DATA['norman']}...")
adata = ad.read_h5ad(DATA['norman'])

print("Overwriting Ensembl index with uppercase HGNC symbols...")
# Force the gene_name column to string, uppercase it, and set it as the index
adata.var_names = adata.var['gene_name'].astype(str).str.upper()
adata.var_names_make_unique() # Prevent duplicate index crashes

clean_norman_path = DATA['norman'].replace('.h5ad', '_clean.h5ad')
print(f"Saving cleaned dataset to {clean_norman_path}...")
adata.write_h5ad(clean_norman_path)
print("Success. Ready for the pipeline.")

Loading raw Norman dataset from data/fibroblast_CRISPRa_custom_mean.h5ad...
Overwriting Ensembl index with uppercase HGNC symbols...


KeyError: 'gene_name'

## 11. Head-to-head: binding-B vs Norman-B against `b_OSKM_effect`
The DECISIVE test. For each of OSKM:
- `cos(b_TF_binding, b_OSKM_effect)`
- `cos(b_TF_norman,  b_OSKM_effect)`

And for the combined OSKM cocktail (the experimental input was all four
together, so this is the most direct test):
- `cos(sum_{OSKM} b_TF_binding, b_OSKM_effect)`
- `cos(sum_{OSKM} b_TF_norman,  b_OSKM_effect)`

Higher cosine = the constructed B is closer to the data-grounded ground truth.

In [ ]:
# Patched head-to-head: restrict to rows where Norman-B has any signal
def _cos(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    return float(a @ b / (na * nb + 1e-12))

OSKM = ['Pou5f1', 'Sox2', 'Klf4', 'Myc']

# rows where Norman-B carries signal = rows mapped via orthology
mask = np.any(np.abs(B_norman) > 1e-12, axis=1)
n_mapped = int(mask.sum())
print(f"intersection: {n_mapped}/{len(mouse_panel_symbols)} panel rows have Norman signal")

def col_idx(tf_names, tf):
    if tf in tf_names: return tf_names.index(tf)
    tf_up = tf.upper()
    for i, t in enumerate(tf_names):
        if str(t).upper() == tf_up: return i
    return None

b_gt = b_OSKM_effect[mask]                 # restrict ground truth to mapped rows
print(f"||b_OSKM_effect|| full={np.linalg.norm(b_OSKM_effect):.3f}  "
      f"on intersection={np.linalg.norm(b_gt):.3f}  "
      f"({100*np.linalg.norm(b_gt)/(np.linalg.norm(b_OSKM_effect)+1e-12):.0f}% of signal)")

print(f"\n{'TF':>8}  {'cos(binding,gt)|int':>20}  {'cos(norman,gt)|int':>20}  winner")
b_bind_sum = np.zeros(len(b_gt)); b_norm_sum = np.zeros(len(b_gt))
for tf in OSKM:
    jb = col_idx(tf_names_binding, tf)
    if jb is not None:
        cb = _cos(B_binding[mask, jb], b_gt); b_bind_sum += B_binding[mask, jb]
    else:
        cb = np.nan
    jn = col_idx(tf_names_norman, tf)
    if jn is not None:
        cn = _cos(B_norman[mask, jn], b_gt); b_norm_sum += B_norman[mask, jn]
    else:
        cn = np.nan
    winner = ('(missing)' if (np.isnan(cb) or np.isnan(cn)) else
              ('NORMAN' if abs(cn) > abs(cb) else 'binding'))
    print(f'{tf:>8}  {cb:>+20.4f}  {cn:>+20.4f}  {winner}')

cb_sum = _cos(b_bind_sum, b_gt); cn_sum = _cos(b_norm_sum, b_gt)
print(f"\ncombined OSKM cocktail (on the {n_mapped}-row intersection):")
print(f"  cos(sum binding-B OSKM, gt) = {cb_sum:+.4f}")
print(f"  cos(sum  norman-B OSKM, gt) = {cn_sum:+.4f}")
print(f"  --> {'NORMAN' if abs(cn_sum) > abs(cb_sum) else 'binding'}")


intersection: 1479/1479 panel rows have Norman signal
||b_OSKM_effect|| full=0.082  on intersection=0.082  (100% of signal)

      TF   cos(binding,gt)|int    cos(norman,gt)|int  winner
  Pou5f1                  +nan               -0.0138  (missing)
    Sox2                  +nan               -0.0191  (missing)
    Klf4                  +nan               +0.0086  (missing)
     Myc                  +nan               -0.0226  (missing)

combined OSKM cocktail (on the 1479-row intersection):
  cos(sum binding-B OSKM, gt) = +0.0000
  cos(sum  norman-B OSKM, gt) = -0.0266
  --> NORMAN


In [ ]:
import scipy.linalg

# 1. Extract the 4D OSKM sub-matrix
oskm_indices = [col_idx(tf_names_norman, tf) for tf in OSKM]
# Ensure none are missing
if None not in oskm_indices:
    B_oskm = B_norman[mask][:, oskm_indices]  # Shape: (1111, 4)
    
    # 2. Orthogonalize the OSKM subspace using QR decomposition
    Q, R = scipy.linalg.qr(B_oskm, mode='economic')
    
    # 3. Project the ground truth vector onto the 4D subspace
    b_proj = Q @ (Q.T @ b_gt)
    
    # 4. Calculate the subspace cosine similarity
    subspace_cos = np.linalg.norm(b_proj) / np.linalg.norm(b_gt)
    
    print(f"\nSubspace Analysis (Optimal Stoichiometry):")
    print(f"  cos(OSKM subspace, gt) = +{subspace_cos:.4f}")
    
    # Bonus: We can actually extract the optimal weights (stoichiometry)
    optimal_weights = np.linalg.lstsq(B_oskm, b_gt, rcond=None)[0]
    print(f"  Optimal TF Weights: O:{optimal_weights[0]:.2f}, S:{optimal_weights[1]:.2f}, K:{optimal_weights[2]:.2f}, M:{optimal_weights[3]:.2f}")
else:
    print("Cannot compute subspace: One or more OSKM factors missing.")


Subspace Analysis (Optimal Stoichiometry):
  cos(OSKM subspace, gt) = +0.0291
  Optimal TF Weights: O:-0.00, S:-0.00, K:-0.00, M:-0.00


In [ ]:
print('b_cache keys:', list(np.load(DATA['b_cache'], allow_pickle=True).keys()))

b_cache keys: ['B', 'tfs']


## 12. Scoring + joint best-day control

In [ ]:
# choose which B to score on; binding by default. swap to B_norman / tf_names_norman
# after Norman is built to see if the ranking improves.
B = B_binding; tf_names = tf_names_binding
tf_filter = [j for j in range(B.shape[1]) if np.linalg.norm(B[:, j]) > 1e-12]
name = lambda j: tf_names[j]

b_scores   = BOnlyScorer().score(B, tf_filter, x_init, x_target)
dyn_scores = DynamicsScorer(seq_dgc, n_steps, timed=True).score(B, tf_filter, x_init, x_target)
res = ControlBestDayScorer(seq_fused, n_steps, eps=0.05).run(B, tf_filter, x_init, x_target)
print(f"best day K={res['best_K']}  gap {res['best_gap']:.2f} -> resid {res['best_resid']:.2f} "
      f"(closed {100*(1-res['best_resid']/res['best_gap']):.0f}% of gap)")
print(f"\n{'K':>3} {'gap':>10} {'resid':>10} {'resid/gap':>10}")
for Kk, gap, resid, frac in res['curve']:
    mark = '  <-- BEST' if Kk == res['best_K'] else ''
    print(f'{Kk:>3} {gap:>10.3f} {resid:>10.3f} {frac:>10.3f}{mark}')

best day K=34  gap 6.56 -> resid 4.88 (closed 26% of gap)

  K        gap      resid  resid/gap
  1     26.522     26.522      1.000
  2     23.524     19.796      0.842
  3     25.866     18.139      0.701
  4     22.981     17.242      0.750
  5     22.697     17.672      0.779
  6     22.091     17.681      0.800
  7     22.016     17.403      0.790
  8     21.619     17.082      0.790
  9     21.152     17.279      0.817
 10     21.018     17.551      0.835
 11     21.145     17.575      0.831
 12     20.767     17.008      0.819
 13     19.809     15.843      0.800
 14     18.757     14.541      0.775
 15     18.127     13.725      0.757
 16     17.762     13.242      0.746
 17     17.324     13.000      0.750
 18     16.677     12.864      0.771
 19     15.935     12.668      0.795
 20     15.211     12.331      0.811
 21     14.510     11.798      0.813
 22     13.798     11.015      0.798
 23     13.117     10.189      0.777
 24     12.535      9.440      0.753
 25     12.050  

## 13. Importance (LOO + mass)

In [ ]:
loo, mass = res['loo'], res['mass']
order = sorted(loo, key=lambda j: -loo[j])
print(f"{'rank':>4} {'TF':>12} {'LOO':>12} {'mass':>12}")
for r, j in enumerate(order[:15], 1):
    print(f'{r:>4} {name(j):>12} {loo[j]:>12.4f} {mass[j]:>12.4f}')

rank           TF          LOO         mass
   1        TF638       0.1482       0.1360
   2        TF609       0.1352       0.0811
   3        TF666       0.0599       0.0580
   4        TF153       0.0357       0.0407
   5         TF50       0.0049       0.0143
   6        TF607       0.0013       0.0020
   7        TF459       0.0007       0.0022
   8          TF7      -0.0000       0.0000
   9         TF19      -0.0000       0.0000
  10         TF20      -0.0000       0.0000
  11         TF21      -0.0000       0.0000
  12         TF24      -0.0000       0.0000
  13         TF35      -0.0000       0.0000
  14         TF40      -0.0000       0.0000
  15         TF48      -0.0000       0.0000


## 15. Verdict checklist
- M_24 vs M_12^2 (cell 5): cos > 0.9 = constant-M model holds at the 24h scale.
- Head-to-head (cell 11): NORMAN winning on the OSKM cocktail = binding-B is the wrong construction; effective-B from perturbations does carry real signal.
- Best-day (cell 12): interior K & OSKM high on LOO = control story holds.
- Fusion vs averaging (cell 14): R^2 gap > 0.05 = the project's fused operator is genuinely better than the trivial baseline.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from tqdm import tqdm

# --- 1. Configuration ---
# Point this to your 9.7 GB single-cell file
SC_FILE = 'data/fibroblast_CRISPRa_final_pop.h5ad'  
OUT_FILE = 'data/fibroblast_CRISPRa_custom_mean.h5ad'

# For the final_pop file, the TF guides are usually stored here:
PERT_COLUMN = 'guide_target' 
MIN_CELLS = 5

print(f"Opening {SC_FILE} in backed mode (memory-efficient)...")
# backed='r' allows us to read metadata without loading the 10GB matrix into RAM
adata_sc = ad.read_h5ad(SC_FILE, backed='r')

# --- 2. Identify Targets ---
# Extract all unique perturbations (including controls)
conditions = adata_sc.obs[PERT_COLUMN].dropna().unique()
print(f"Found {len(conditions)} unique perturbation conditions.")

# --- 3. Chunked Aggregation ---
mean_profiles = {}
valid_conditions = []

print("Aggregating single-cell droplets into pseudo-bulk profiles...")
for cond in tqdm(conditions, desc="Processing TFs"):
    # Find the row indices for all cells that received this specific guide
    # We must use sorted integer indices for fast backed-mode HDF5 slicing
    idx = np.where(adata_sc.obs[PERT_COLUMN] == cond)[0]
    
    if len(idx) < MIN_CELLS:
        continue
        
    # Pull ONLY these specific cells into RAM
    chunk = adata_sc[idx, :].X
    
    # Calculate the mean expression across the cells
    if issparse(chunk):
        mean_vec = np.asarray(chunk.mean(axis=0)).flatten()
    else:
        mean_vec = np.mean(chunk, axis=0).flatten()
        
    mean_profiles[cond] = mean_vec
    valid_conditions.append(cond)

# --- 4. Assemble the New Matrix ---
print("\nAssembling the aggregated AnnData object...")
new_X = np.vstack([mean_profiles[cond] for cond in valid_conditions])

new_obs = pd.DataFrame(index=valid_conditions)
new_obs[PERT_COLUMN] = valid_conditions

# Copy over the original gene metadata
new_var = adata_sc.var.copy()

adata_mean = ad.AnnData(X=new_X, obs=new_obs, var=new_var)

# --- 5. Fix the Index (The Uppercase Trick permanently applied) ---
print("Cleaning the gene index to standard uppercase HGNC symbols...")
if 'gene_name' in adata_mean.var.columns:
    adata_mean.var_names = adata_mean.var['gene_name'].astype(str).str.upper()
else:
    adata_mean.var_names = adata_mean.var_names.astype(str).str.upper()

adata_mean.var_names_make_unique()

# --- 6. Save ---
print(f"Saving fully populated pseudo-bulk matrix to {OUT_FILE}...")
adata_mean.write_h5ad(OUT_FILE)
print("Done! You are ready to run the NormanBBuilder.")

Opening data/fibroblast_CRISPRa_final_pop.h5ad in backed mode (memory-efficient)...
Found 1837 unique perturbation conditions.
Aggregating single-cell droplets into pseudo-bulk profiles...


Processing TFs:  63%|██████▎   | 1159/1837 [01:51<00:54, 12.52it/s]Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x000001C421103060>
Traceback (most recent call last):
  File "c:\Users\28013\AppData\Local\Programs\Python\Python313\Lib\weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 
Processing TFs: 100%|██████████| 1837/1837 [02:37<00:00, 11.63it/s]



Assembling the aggregated AnnData object...
Cleaning the gene index to standard uppercase HGNC symbols...
Saving fully populated pseudo-bulk matrix to data/fibroblast_CRISPRa_custom_mean.h5ad...


ValueError: DataFrame.index.name ('gene_name') is also used by a column whose values are different. This is not supported. Please make sure the values are the same, or use a different name.